We import our Dependencies into our Code

In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt


We take the Archives from the data folder and store them in each variable (this dataset is from kaggle and it already is separated in train and test data),The data is already shuffled and I gonna leave the URL in the README.md file.

In [ ]:
train_df = pd.read_csv("data/mnist_train.csv")
test_df = pd.read_csv("data/mnist_test.csv")

We separate them again taking the column labeled away from the columns of the pixels so each pixels row have a target which is in the column "label" of the respective row.

In [ ]:
inputs_train = train_df.drop(columns=["label"]).to_numpy()
targets_train = train_df["label"].to_numpy()

inputs_test = test_df.drop(columns=["label"]).to_numpy()
targets_test = test_df["label"].to_numpy()

This step is called "Normalization",cause the neural network works better with numbers between 0-1 and the pixels have values between 0-255

In [ ]:
inputs_train = inputs_train / 255.0
inputs_test = inputs_test / 255.0

The next step would be the creation of batches,cause we don't wanna do a simple foward pass with the whole train data of 60000 images, it would be inefficent in terms of GPU,Time, and noise in the gradient(it makes the gradient "too perfect",No Noise,)making the learning worse.

We don't want a batch too tiny either(Ex= 1 Image),that will make the training unstable,the gradient too noisy and a training incredible slow.

For that reason the batches will be of 64 images each in this case.

In [ ]:
def create_batches(inputs_train, targets_train, batch_size=64, seed=42):
    # For replication of the shuffle.
    np.random.seed(seed)

    n_samples = inputs_train.shape[0]

    #i've seen the data and is already shuffled but i'll do it again,just for learning purposes.
    indices = np.arange(n_samples)
    np.random.shuffle(indices)

    for start in range(0,n_samples,batch_size):
        
        end = start + batch_size

        batch_idx = indices[start:end]
        
        yield inputs_train[batch_idx], targets_train[batch_idx]

Now the nexts step should be in order for the code to work.

First,the initilization of the parameters.In the first foward pass we need to assign random custom values to the weights and bias ( in the case of the bias they usually starts at zero)then them will be modified according to the back propagation.

The use of different formulas in the weights is because of the Variance required for the formula in the hidden layer("ReLU") and the formula of the output layer("Softmax")

ReLU needs more variance cause the formula is inclined to "kill" the neuron,and more variance will solve the problem.\
Softmax need less variance otherwise it get saturated.



In [ ]:
def init_parameters(input_layer,hidden_layer,output_layer):
    
    W1 = np.random.randn(input_layer,hidden_layer) * np.sqrt(2 / input_layer)#Multiplied by "He init" formula
    B1 = np.zeros((1,hidden_layer))
    W2 = np.random.randn(hidden_layer,output_layer) * np.sqrt(1 / hidden_layer)#Multiplied by "Xavier" formula
    B2 = np.zeros((1,output_layer))

    return W1,B1,W2,B2
    

We call the function starting the parameters and giving for arguments the size of each layer of our Neural Network

Input layer = 784\
Hidden layer = 32\
Output Layer = 10



In [ ]:
W1,B1,W2,B2 = init_parameters(784,32,10)

The Second Step is the Forward pass,we make each image of each batch go through every layer appling the respective formula of each layer.

In the hidden layer we use the "ReLU" formula that will make the number zero or below to zero return 0,and numbers above zero return its value, separating the pixel that have activated away for the ones who don't.

And in the output layer we have the "Softmax" formula that will make every of the 10 rows of the output layer have a value between 0-1 each.Value that later we gonna interpreted as percentage (0%-100%),giving a prediction of the most likely correct answer.

In [ ]:
def Foward_Pass(train_batch,W1,B1,W2,B2):
    
    Z1 = np.dot(train_batch,W1)+ B1
    
    A1 = np.maximum(0,Z1) #ReLU

    Z2 = np.dot(A1,W2)+B2

    exp_scores = np.exp(Z2 - np.max(Z2, axis=1, keepdims=True))# Softmax
    A2 = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)#

    return Z1,A1,Z2,A2

We made the One Hot Encoding of the labels for the reason of making the matrix that the neural network should replicate.The purpose of the neural network is aproximate the closest possible with the prediction matrix of the output layer to the matrix of the one hot encoding.

In [13]:
def One_hot_encoding(train_batch):
    
    onehot = np.zeros((train_batch.size,10))
    
    onehot[np.arange(train_batch.size),train_batch] = 1

    return onehot